# Phase 2 — run the pipeline on Databricks

Import this into Databricks and run top to bottom. It does steps 3–5:
land the raw data in the lakehouse, build Bronze and Silver with **Delta**,
then run everything a second time to prove idempotency.

Attach to serverless compute. Roughly 10-20 minutes: the slow part is
Bronze over the 267k StatsBomb events, not the ingest, since the raw data
is copied from the Repo when it is already there rather than downloaded.

## 0. Configuration

In [0]:
import os, sys, subprocess

REPO = "/Workspace/Repos/abdullahazmat.w@gmail.com/football-analytics-lakehouse"

assert os.path.exists(REPO), f"repo not found at {REPO} - fix REPO above"
if REPO not in sys.path:
    sys.path.insert(0, REPO)

# Drop any previously-imported pipeline modules.
#
# Databricks keeps the Python interpreter alive between runs, so once
# `pipelines.*` has been imported it stays in sys.modules. Pulling new code
# from git changes the files on disk but NOT the modules already in memory, so
# a re-run silently executes the old code and you debug a bug you already
# fixed. Purging them here forces a genuine re-read from disk.
_stale = [m for m in sys.modules if m == "pipelines" or m.startswith("pipelines.")]
for m in _stale:
    del sys.modules[m]
if _stale:
    print(f"purged {len(_stale)} cached pipeline modules so the pulled code is used")

# Where the lakehouse lives.
#
# Newer workspaces (Free Edition included) have the public DBFS root switched
# off - writing to dbfs:/FileStore fails with DBFS_DISABLED. The supported
# replacement is a Unity Catalog Volume, which has the convenient property that
# Spark and plain python file IO use the SAME path, so there is no dbfs: vs
# /dbfs/ split to get wrong.
#
# This finds a catalog and schema you can actually write to, creates the volume
# if needed, and falls back to DBFS only on older workspaces where that works.

VOLUME_NAME = "football_lakehouse"


def _first_writable_catalog_schema():
    prefer_catalogs = ["workspace", "main"]
    catalogs = [r[0] for r in spark.sql("SHOW CATALOGS").collect()]  # noqa: F821
    ordered = [c for c in prefer_catalogs if c in catalogs] + \
              [c for c in catalogs if c not in prefer_catalogs and
               c not in ("samples", "system", "hive_metastore")]
    for cat in ordered:
        try:
            schemas = [r[0] for r in spark.sql(f"SHOW SCHEMAS IN `{cat}`").collect()]  # noqa: F821
        except Exception:
            continue
        for sch in (["default"] + [s for s in schemas if s != "default"]):
            if sch in schemas and sch != "information_schema":
                return cat, sch
    return None, None


LAKE = None
try:
    cat, sch = _first_writable_catalog_schema()
    if cat:
        spark.sql(f"CREATE VOLUME IF NOT EXISTS `{cat}`.`{sch}`.`{VOLUME_NAME}`")  # noqa: F821
        LAKE = f"/Volumes/{cat}/{sch}/{VOLUME_NAME}"
        print(f"using Unity Catalog volume: {LAKE}")
except Exception as e:  # noqa: BLE001
    print(f"could not create a UC volume ({e}); trying DBFS")

if LAKE is None:
    LAKE = "dbfs:/FileStore/football_lakehouse"
    print(f"falling back to DBFS: {LAKE}")

os.environ["LAKE_ROOT"] = LAKE
os.environ["TABLE_FORMAT"] = "delta"          # the path that has never been run

print("repo :", REPO)
print("lake :", LAKE)

using Unity Catalog volume: /Volumes/workspace/default/football_lakehouse
repo : /Workspace/Repos/abdullahazmat.w@gmail.com/football-analytics-lakehouse
lake : /Volumes/workspace/default/football_lakehouse


## 1. Land the raw data in the lakehouse

Whether the committed data survives the Repos clone varies by workspace
and file size, so this detects rather than assumes:

* **data present in the Repo** - copy it across (seconds, no network)
* **data absent** - download it (~233 MB, several minutes)

Either way the pipeline reads from the lakehouse root, so everything
downstream is identical.

In [0]:
# On a UC Volume both Spark and python file IO use the same path. On DBFS they
# differ (dbfs:/x for Spark, /dbfs/x for python), so derive both explicitly.
RAW_URI = f"{LAKE}/raw"                                   # for Spark
RAW = RAW_URI.replace("dbfs:/", "/dbfs/", 1) if RAW_URI.startswith("dbfs:/") else RAW_URI
print("raw (spark) :", RAW_URI)
print("raw (python):", RAW)


def dir_bytes(p: str) -> int:
    out = subprocess.run(["du", "-sb", p], capture_output=True, text=True).stdout
    try:
        return int(out.split()[0])
    except Exception:
        return 0


sb_local = f"{REPO}/data/statsbomb"
fd_local = f"{REPO}/data/full_load"
sb_have = dir_bytes(sb_local) > 100_000_000      # expect ~233 MB
fd_have = dir_bytes(fd_local) > 10_000_000       # expect ~16 MB

print(f"StatsBomb in repo    : {dir_bytes(sb_local)/1e6:7.1f} MB  -> {'COPY' if sb_have else 'DOWNLOAD'}")
print(f"football-data in repo: {dir_bytes(fd_local)/1e6:7.1f} MB  -> {'COPY' if fd_have else 'DOWNLOAD'}")

raw (spark) : /Volumes/workspace/default/football_lakehouse/raw
raw (python): /Volumes/workspace/default/football_lakehouse/raw
StatsBomb in repo    :   232.8 MB  -> COPY
football-data in repo:    15.9 MB  -> COPY


In [0]:
if sb_have:
    dbutils.fs.cp(f"file:{sb_local}", f"{RAW_URI}/statsbomb", recurse=True)  # noqa: F821
    print("copied StatsBomb from the repo")
else:
    r = subprocess.run([sys.executable, f"{REPO}/scripts/ingest_statsbomb.py",
                        "--out", f"{RAW}/statsbomb"],
                       capture_output=True, text=True, cwd=REPO)
    print(r.stdout[-3000:]); print(r.stderr[-2000:] if r.returncode else "")
    assert r.returncode == 0, "StatsBomb ingest failed"

copied StatsBomb from the repo


In [0]:
if fd_have:
    dbutils.fs.cp(f"file:{fd_local}", f"{RAW_URI}/football-data/full_load", recurse=True)  # noqa: F821
    print("copied football-data full load from the repo")
else:
    # Same bytes as the committed files (checksum-verified), fetched over
    # raw.githubusercontent so no API key and no rate limit are involved.
    r = subprocess.run([sys.executable, f"{REPO}/scripts/fetch_committed_raw.py",
                        "--out", f"{RAW}/football-data/full_load"],
                       capture_output=True, text=True, cwd=REPO)
    print(r.stdout[-2000:]); print(r.stderr[-1000:] if r.returncode else "")
    assert r.returncode == 0, "football-data fetch failed"

copied football-data full load from the repo


In [0]:
# confirm what actually landed before building anything on top of it
for sub, expect in [("statsbomb/events", 66), ("football-data/full_load", 76)]:
    n = len(dbutils.fs.ls(f"{RAW_URI}/{sub}"))  # noqa: F821
    print(f"{sub:28} {n:>4} files   {'OK' if n >= expect else f'EXPECTED >= {expect}  <-- PROBLEM'}")

statsbomb/events               66 files   OK
football-data/full_load        76 files   OK


### Optional: a live daily snapshot

Only needed to demonstrate the incremental load against the live API.
Store the key as a secret first - do not paste it into the notebook:
```
databricks secrets create-scope football
databricks secrets put-secret football api_token
```
Takes ~5 minutes (rate limited to 10 req/min). Skip it if you only need
the full load.

In [0]:
try:
    os.environ["FOOTBALL_DATA_TOKEN"] = dbutils.secrets.get("football", "api_token")  # noqa: F821
    r = subprocess.run([sys.executable, f"{REPO}/scripts/daily_snapshot.py",
                        "--out", f"{RAW}/football-data/daily"],
                       capture_output=True, text=True, cwd=REPO, timeout=1800)
    print(r.stdout[-3000:])
except Exception as e:  # noqa: BLE001
    print(f"skipped daily snapshot ({e}) - the full load above is enough to run the pipeline")

skipped daily snapshot (An error occurred while calling GetSecret.
: java.lang.IllegalArgumentException: Secret does not exist with scope: football and key: api_token
	at com.databricks.backend.common.rpc.SimpleSecretManagerClient.getSecret(SecretManagerClient.scala:291)
	at com.databricks.dbutils_v1.impl.SecretUtilsImpl.getBytes(SecretUtilsImpl.scala:104)
	at com.databricks.dbutils_v1.impl.SecretUtilsImpl.get(SecretUtilsImpl.scala:89)
	at com.databricks.backend.daemon.driver.PythonDriverLocalBase$PythonDriverLocalGrpcInterfaceImpl.getSecret(PythonDriverLocalBase.scala:1172)
	at com.databricks.backend.daemon.repl.CredentialsService.$anonfun$getSecret$2(CredentialsService.scala:154)
	at com.databricks.backend.daemon.repl.CredentialsService.translateUserErrors(CredentialsService.scala:45)
	at com.databricks.backend.daemon.repl.CredentialsService.$anonfun$getSecret$1(CredentialsService.scala:151)
	at com.databricks.backend.daemon.repl.NotebookGrpcServiceBase.$anonfun$handleRequestAsyncWit

## 2. Build Bronze and Silver (Delta)

In [0]:
from pipelines.bronze import raw_to_bronze
from pipelines.silver import bronze_to_silver
from pipelines.common.audit import new_batch_id
from pipelines.common import config
from pyspark.sql import functions as F

SNAPSHOT_DATE = "2026-09-27"   # the date the committed full load was pulled

BRONZE_JOBS = [
    ("fd_matches",   f"{RAW_URI}/football-data/full_load/matches_*.json",   "FULL", None),
    ("fd_standings", f"{RAW_URI}/football-data/full_load/standings_*.json", "FULL", SNAPSHOT_DATE),
    ("fd_scorers",   f"{RAW_URI}/football-data/full_load/scorers_*.json",   "FULL", SNAPSHOT_DATE),
    ("fd_teams",     f"{RAW_URI}/football-data/full_load/teams_*.json",     "FULL", None),
    ("sb_matches",   f"{RAW_URI}/statsbomb/matches/*/*.json",               "FULL", None),
    ("sb_lineups",   f"{RAW_URI}/statsbomb/lineups/*.json",                 "FULL", None),
    ("sb_events",    f"{RAW_URI}/statsbomb/events/*.json",                  "FULL", None),
]
SILVER_JOBS = [
    ("silver_matches", None), ("silver_standings", SNAPSHOT_DATE),
    ("silver_scorers", SNAPSHOT_DATE), ("silver_teams", None),
    ("silver_sb_matches", None), ("silver_lineups", None), ("silver_events", None),
]


def run_everything(tag: str):
    b = new_batch_id(f"bronze_{tag}")
    for feed, path, lt, snap in BRONZE_JOBS:
        raw_to_bronze.run_feed(spark, feed, path, lt, b, snap)  # noqa: F821
    s = new_batch_id(f"silver_{tag}")
    for tbl, snap in SILVER_JOBS:
        bronze_to_silver.run_table(spark, tbl, s, "FULL", snap)  # noqa: F821


def snapshot_counts():
    out = {}
    for layer, tables in (("bronze", [t for _, t in
                                      sorted(raw_to_bronze.TARGET_TABLE.items())]),
                          ("silver", [t for t, _ in SILVER_JOBS])):
        for t in tables:
            try:
                out[t] = spark.read.format("delta").load(  # noqa: F821
                    config.table_path(layer, t)).count()
            except Exception:
                out[t] = None
    return out

In [0]:
print("=" * 60); print("PASS 1"); print("=" * 60)
run_everything("p1")
counts_1 = snapshot_counts()

PASS 1
[bronze_fd_matches] read=14001 inserted=0 updated=14001 quarantined=0
[bronze_fd_standings] read=212 inserted=0 updated=212 quarantined=0
[bronze_fd_scorers] read=600 inserted=0 updated=600 quarantined=0
[bronze_fd_teams] read=284 inserted=0 updated=284 quarantined=0
[bronze_sb_matches] read=66 inserted=0 updated=66 quarantined=0
[bronze_sb_lineups] read=2620 inserted=0 updated=2620 quarantined=0
[bronze_sb_events] read=267255 inserted=0 updated=267255 quarantined=0
[silver_matches] read=14001 inserted=0 updated=14001 quarantined=0
[silver_standings] read=212 inserted=0 updated=212 quarantined=0
[silver_scorers] read=600 inserted=0 updated=600 quarantined=0
[silver_teams] read=284 inserted=0 updated=284 quarantined=0
[silver_sb_matches] read=66 inserted=0 updated=66 quarantined=0
[silver_lineups] read=2620 inserted=0 updated=2620 quarantined=0
[silver_events] read=267255 inserted=0 updated=267255 quarantined=0


## 3. Idempotency — run it all again

Identical input, fresh batch ids. Row counts must not move, and the
second pass must report `inserted=0, updated=N`. This is what exercises
Delta `MERGE INTO`.

In [0]:
print("=" * 60); print("PASS 2"); print("=" * 60)
run_everything("p2")
counts_2 = snapshot_counts()

PASS 2
[bronze_fd_matches] read=14001 inserted=0 updated=14001 quarantined=0
[bronze_fd_standings] read=212 inserted=0 updated=212 quarantined=0
[bronze_fd_scorers] read=600 inserted=0 updated=600 quarantined=0
[bronze_fd_teams] read=284 inserted=0 updated=284 quarantined=0
[bronze_sb_matches] read=66 inserted=0 updated=66 quarantined=0
[bronze_sb_lineups] read=2620 inserted=0 updated=2620 quarantined=0
[bronze_sb_events] read=267255 inserted=0 updated=267255 quarantined=0
[silver_matches] read=14001 inserted=0 updated=14001 quarantined=0
[silver_standings] read=212 inserted=0 updated=212 quarantined=0
[silver_scorers] read=600 inserted=0 updated=600 quarantined=0
[silver_teams] read=284 inserted=0 updated=284 quarantined=0
[silver_sb_matches] read=66 inserted=0 updated=66 quarantined=0
[silver_lineups] read=2620 inserted=0 updated=2620 quarantined=0
[silver_events] read=267255 inserted=0 updated=267255 quarantined=0


In [0]:
ok = True
print(f"{'table':26}{'pass1':>10}{'pass2':>10}   idempotent")
for t in counts_1:
    a, b = counts_1[t], counts_2.get(t)
    good = a is not None and a == b
    ok = ok and good
    print(f"{t:26}{str(a):>10}{str(b):>10}   {'YES' if good else 'NO  <-- FAIL'}")
print()
print("IDEMPOTENCY (Delta MERGE):", "PASS" if ok else "FAIL")

table                          pass1     pass2   idempotent
bronze_fd_matches              14001     14001   YES
bronze_fd_scorers                600       600   YES
bronze_fd_standings              212       212   YES
bronze_fd_teams                  284       284   YES
bronze_sb_events              267255    267255   YES
bronze_sb_lineups               2620      2620   YES
bronze_sb_matches                 66        66   YES
silver_matches                 14001     14001   YES
silver_standings                 212       212   YES
silver_scorers                   600       600   YES
silver_teams                     284       284   YES
silver_sb_matches                 66        66   YES
silver_lineups                  2620      2620   YES
silver_events                 267255    267255   YES

IDEMPOTENCY (Delta MERGE): PASS


## 4. Audit log — evidence for submission

In [0]:
log = spark.read.format("delta").load(config.EXECUTION_LOG_PATH)  # noqa: F821
print("total audit rows:", log.count())
log.groupBy("layer", "load_type", "status").count().orderBy("layer").display()

total audit rows: 59


layer,load_type,status,count
Bronze-to-Silver,FULL,SUCCESS,28
Raw-to-Bronze,FULL,FAILED,3
Raw-to-Bronze,FULL,SUCCESS,28


In [0]:
(log.select("layer", "table_name", "parameter", "load_type", "status",
            "rows_read", "rows_inserted", "rows_updated", "rows_quarantined",
            "start_time", "end_time", "duration_seconds")
    .orderBy("start_time")
    .display())

layer,table_name,parameter,load_type,status,rows_read,rows_inserted,rows_updated,rows_quarantined,start_time,end_time,duration_seconds
Raw-to-Bronze,bronze_fd_matches,/Volumes/workspace/default/football_lakehouse/raw/football-data/full_load/matches_*.json,FULL,FAILED,0,0,0,0,2026-10-10T11:32:53.724Z,2026-10-10T11:33:02.249Z,8
Raw-to-Bronze,bronze_fd_matches,/Volumes/workspace/default/football_lakehouse/raw/football-data/full_load/matches_*.json,FULL,FAILED,0,0,0,0,2026-10-10T11:43:14.715Z,2026-10-10T11:43:16.737Z,2
Raw-to-Bronze,bronze_fd_matches,/Volumes/workspace/default/football_lakehouse/raw/football-data/full_load/matches_*.json,FULL,FAILED,0,0,0,0,2026-10-10T11:50:41.001Z,2026-10-10T11:50:48.178Z,7
Raw-to-Bronze,bronze_fd_matches,/Volumes/workspace/default/football_lakehouse/raw/football-data/full_load/matches_*.json,FULL,SUCCESS,14001,14001,0,0,2026-10-10T11:59:27.767Z,2026-10-10T11:59:36.329Z,8
Raw-to-Bronze,bronze_fd_standings,/Volumes/workspace/default/football_lakehouse/raw/football-data/full_load/standings_*.json,FULL,SUCCESS,212,212,0,0,2026-10-10T11:59:37.653Z,2026-10-10T11:59:45.295Z,7
Raw-to-Bronze,bronze_fd_scorers,/Volumes/workspace/default/football_lakehouse/raw/football-data/full_load/scorers_*.json,FULL,SUCCESS,600,600,0,0,2026-10-10T11:59:46.693Z,2026-10-10T11:59:52.966Z,6
Raw-to-Bronze,bronze_fd_teams,/Volumes/workspace/default/football_lakehouse/raw/football-data/full_load/teams_*.json,FULL,SUCCESS,284,284,0,0,2026-10-10T11:59:54.187Z,2026-10-10T12:00:01.792Z,7
Raw-to-Bronze,bronze_sb_matches,/Volumes/workspace/default/football_lakehouse/raw/statsbomb/matches/*/*.json,FULL,SUCCESS,66,66,0,0,2026-10-10T12:00:03.078Z,2026-10-10T12:00:08.899Z,5
Raw-to-Bronze,bronze_sb_lineups,/Volumes/workspace/default/football_lakehouse/raw/statsbomb/lineups/*.json,FULL,SUCCESS,2620,2620,0,0,2026-10-10T12:00:10.198Z,2026-10-10T12:00:17.226Z,7
Raw-to-Bronze,bronze_sb_events,/Volumes/workspace/default/football_lakehouse/raw/statsbomb/events/*.json,FULL,SUCCESS,267255,267255,0,0,2026-10-10T12:00:18.421Z,2026-10-10T12:00:29.401Z,10


Screenshot the two tables above for the submission: they show every run,
its parameter, timings, status and row counts, and the second pass
updating rather than inserting.

In [0]:
failed = log.filter("status != 'SUCCESS'")
if failed.count():
    failed.select("table_name", "parameter", "error_message").display()
else:
    print("no failed runs")

table_name parameter error_message bronze_fd_matches /Volumes/workspace/default/football_lakehouse/raw/football-data/full_load/matches_*.json AnalysisException: [UC_COMMAND_NOT_SUPPORTED.WITH_RECOMMENDATION] The command(s): input_file_name are not supported in Unity Catalog. Please use _metadata.file_path instead. SQLSTATE: 0AKUC;
Aggregate [count(1) AS count(1)#10208L]
+- Project [match_id#10140L, competition_code#10141, competition_id#10142L, competition_name#10143, season_id#10144L, season_start_date#10145, season_end_date#10146, season_current_matchday#10147, utc_date#10148, status#10149, matchday#10150, stage#10151, group_name#10152, source_last_updated#10153, home_team_id#10154L, home_team_name#10155, home_team_tla#10156, away_team_id#10157L, away_team_name#10158, away_team_tla#10159, score_winner#10160, score_duration#10161, full_time_home#10162, full_time_away#10163, half_time_home#10164, ... 6 more fields]
 +- Project [match_id#10140L, competition_code#10141, competition_id#10142L, competition_name#10143, season_id#10144L, season_start_date#10145, season_end_date#10146, season_current_matchday#10147, utc_date#10148, status#10149, matchday#10150, stage#10151, group_name#10152, source_last_updated#10153, home_team_id#10154L, home_team_name#10155, home_team_tla#10156, away_team_id#10157L, away_team_name#10158, away_team_tla#10159, score_winner#10160, score_duration#10161, full_time_home#10162, full_time_away#10163, half_time_home#10164, ... 5 more fields]
 +- Project [match_id#10140L, competition_code#10141, competition_id#10142L, competition_name#10143, season_id#10144L, season_start_date#10145, season_end_date#10146, season_current_matchday#10147, utc_date#10148, status#10149, matchday#10150, stage#10151, group_name#10152, source_last_updated#10153, home_team_id#10154L, home_team_name#10155, home_team_tla#10156, away_team_id#10157L, away_team_name#10158, away_team_tla#10159, score_winner#10160, score_duration#10161, full_time_home#10162, full_time_away#10163, half_time_home#10164, ... 4 more fields]
 +- Project [match_id#10140L, competition_code#10141, competition_id#10142L, competition_name#10143, season_id#10144L, season_start_date#10145, season_end_date#10146, season_current_matchday#10147, utc_date#10148, status#10149, matchday#10150, stage#10151, group_name#10152, source_last_updated#10153, home_team_id#10154L, home_team_name#10155, home_team_tla#10156, away_team_id#10157L, away_team_name#10158, away_team_tla#10159, score_winner#10160, score_duration#10161, full_time_home#10162, full_time_away#10163, half_time_home#10164, ... 3 more fields]
 +- Project [m#10168.id AS match_id#10140L, coalesce(m#10168.competition.code, file_competition_code#10139) AS competition_code#10141, m#10168.competition.id AS competition_id#10142L, m#10168.competition.name AS competition_name#10143, m#10168.season.id AS season_id#10144L, m#10168.season.startDate AS season_start_date#10145, m#10168.season.endDate AS season_end_date#10146, m#10168.season.currentMatchday AS season_current_matchday#10147, m#10168.utcDate AS utc_date#10148, m#10168.status AS status#10149, m#10168.matchday AS matchday#10150, m#10168.stage AS stage#10151, m#10168.group AS group_name#10152, m#10168.lastUpdated AS source_last_updated#10153, m#10168.homeTeam.id AS home_team_id#10154L, m#10168.homeTeam.name AS home_team_name#10155, m#10168.homeTeam.tla AS home_team_tla#10156, m#10168.awayTeam.id AS away_team_id#10157L, m#10168.awayTeam.name AS away_team_name#10158, m#10168.awayTeam.tla AS away_team_tla#10159, m#10168.score.winner AS score_winner#10160, m#10168.score.duration AS score_duration#10161, m#10168.score.fullTime.home AS full_time_home#10162, m#10168.score.fullTime.away AS full_time_away#10163, m#10168.score.halfTime.home AS half_time_home#10164, ... 2 more fields]
 +- Project [m#10168, competition#10135.code AS file_competition_code#10139]
 +- Generate explode(matches#10137), false, [m#10168]
 +- Relation [competition#10135,resultSet#10136,matches#10137] json

## 5. Quarantine (should be empty on clean data)

In [0]:
for t in ["bronze_fd_matches", "silver_matches", "silver_events"]:
    try:
        q = spark.read.format("delta").load(config.table_path("quarantine", t))  # noqa: F821
        print(f"{t}: {q.count()} quarantined")
    except Exception:
        print(f"{t}: no quarantine table (nothing rejected)")

bronze_fd_matches: no quarantine table (nothing rejected)
silver_matches: no quarantine table (nothing rejected)
silver_events: no quarantine table (nothing rejected)


# 6. Proof for the rubric

Sections 2-5 prove the full load, Delta MERGE idempotency and the audit
log. The rubric also wants to *see*, running in this workspace:

| # | What the rubric asks for | Where |
|---|---|---|
| 6.1 | explicit schemas, not inferred | prints every `StructType` |
| 6.2 | parameters and a backfill | reprocesses one competition by `--path` |
| 6.3 | full **and incremental** loads | an INCREMENTAL snapshot load |
| 6.4 | how a schema change is handled | corrupt file, live, into quarantine |
| 6.5 | the log table, both load types | final audit summary |

Everything below is additive and safe to re-run.

## 6.1 Explicit schemas

Every raw feed is read with a declared `StructType`. Nothing calls
`inferSchema`, so a source that changes type cannot silently change the
table's type underneath us - the value fails the cast and is quarantined
instead (6.4 shows that happening).

In [0]:
from pipelines.common.schemas import RAW_SCHEMAS, ARRAY_ROOTED
from pyspark.sql import functions as F

print("grep for inferSchema in the pipeline package:")
_hits = subprocess.run(["grep", "-rn", "inferSchema", f"{REPO}/pipelines"],
                       capture_output=True, text=True).stdout.strip()
print("   none found - every read is schema-on-read" if not _hits else _hits)
print()
print(f"{len(RAW_SCHEMAS)} declared raw schemas:\n")
for feed, sch in RAW_SCHEMAS.items():
    root = "array-rooted file" if feed in ARRAY_ROOTED else "object-rooted file"
    print(f"--- {feed}  ({root}, {len(sch.fields)} top-level fields) ---")
    print(sch.simpleString()[:600] + ("..." if len(sch.simpleString()) > 600 else ""))
    print()

grep for inferSchema in the pipeline package:
/Workspace/Repos/abdullahazmat.w@gmail.com/football-analytics-lakehouse/pipelines/bronze/raw_to_bronze.py:37:    """Read with an explicit schema. inferSchema is never used.
/Workspace/Repos/abdullahazmat.w@gmail.com/football-analytics-lakehouse/pipelines/common/schemas.py:3:Requirement: strict schema-on-read. Spark's inferSchema is never used anywhere

7 declared raw schemas:

--- fd_matches  (object-rooted file, 3 top-level fields) ---
struct<competition:struct<id:bigint,name:string,code:string,type:string,emblem:string>,resultSet:struct<count:int,first:string,last:string,played:int>,matches:array<struct<id:bigint,utcDate:string,status:string,matchday:int,stage:string,group:string,lastUpdated:string,area:struct<id:bigint,name:string>,competition:struct<id:bigint,name:string,code:string,type:string,emblem:string>,season:struct<id:bigint,startDate:string,endDate:string,currentMatchday:int,winner:string>,homeTeam:struct<id:bigint,name:string,

In [0]:
# the full tree for one feed, and the Delta schema it produces.
# An empty frame built from the declared StructType prints the schema we
# *demand* of the source, independently of any data.
print("DECLARED schema for the football-data matches feed:")
spark.createDataFrame([], RAW_SCHEMAS["fd_matches"]).printSchema()  # noqa: F821

print("RESULTING Delta table schema (bronze_fd_matches):")
spark.read.format("delta").load(config.table_path("bronze", "bronze_fd_matches")).printSchema()  # noqa: F821

DECLARED schema for the football-data matches feed:
root
 |-- competition: struct (nullable = true)
 |    |-- id: long (nullable = true)
 |    |-- name: string (nullable = true)
 |    |-- code: string (nullable = true)
 |    |-- type: string (nullable = true)
 |    |-- emblem: string (nullable = true)
 |-- resultSet: struct (nullable = true)
 |    |-- count: integer (nullable = true)
 |    |-- first: string (nullable = true)
 |    |-- last: string (nullable = true)
 |    |-- played: integer (nullable = true)
 |-- matches: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- id: long (nullable = true)
 |    |    |-- utcDate: string (nullable = true)
 |    |    |-- status: string (nullable = true)
 |    |    |-- matchday: integer (nullable = true)
 |    |    |-- stage: string (nullable = true)
 |    |    |-- group: string (nullable = true)
 |    |    |-- lastUpdated: string (nullable = true)
 |    |    |-- area: struct (nullable = true)
 |    |    |    |

## 6.2 Parameters and a backfill

Both loaders are command-line programs: `--feed/--table`, `--path`,
`--load-type`, `--batch-id`, `--snapshot-date`. A backfill is therefore
just the same loader pointed at a narrower path, which is what makes
"re-run one competition for one season" possible without touching the
rest of the table.

Below: reprocess **Premier League 2023 only**, with a batch id we choose.
The table total must not move (MERGE updates the 380 matching rows in
place) and the audit row must carry our batch id and our path.

In [0]:
BACKFILL_PATH = f"{RAW_URI}/football-data/full_load/matches_PL_2023.json"
BACKFILL_BATCH = "backfill_PL_2023_demo"

bronze_matches = config.table_path("bronze", "bronze_fd_matches")
before = spark.read.format("delta").load(bronze_matches).count()  # noqa: F821
print(f"bronze_fd_matches before backfill: {before:,} rows")

argv = ["--feed", "fd_matches", "--path", BACKFILL_PATH,
        "--load-type", "FULL", "--batch-id", BACKFILL_BATCH]
print("\n$ python -m pipelines.bronze.raw_to_bronze " + " ".join(argv) + "\n")
try:
    raw_to_bronze.main(argv)
except Exception as e:  # noqa: BLE001
    # Serverless refuses some SparkSession.builder options on an existing
    # session. The argument handling is the thing being demonstrated, so fall
    # back to the function the CLI calls, with the same parsed values.
    print(f"(CLI entry point unavailable here: {e})")
    print("running the same call through run_feed with identical parameters")
    raw_to_bronze.run_feed(spark, "fd_matches", BACKFILL_PATH, "FULL", BACKFILL_BATCH, None)  # noqa: F821

after = spark.read.format("delta").load(bronze_matches).count()  # noqa: F821
print(f"\nbronze_fd_matches after backfill : {after:,} rows")
print("BACKFILL IS NON-DESTRUCTIVE:", "PASS" if after == before else f"FAIL (moved by {after-before})")

bronze_fd_matches before backfill: 14,001 rows

$ python -m pipelines.bronze.raw_to_bronze --feed fd_matches --path /Volumes/workspace/default/football_lakehouse/raw/football-data/full_load/matches_PL_2023.json --load-type FULL --batch-id backfill_PL_2023_demo

[bronze_fd_matches] read=380 inserted=0 updated=380 quarantined=0

bronze_fd_matches after backfill : 14,001 rows
BACKFILL IS NON-DESTRUCTIVE: PASS


In [0]:
# the audit row for that backfill: our batch id, our path, updates not inserts
(spark.read.format("delta").load(config.EXECUTION_LOG_PATH)  # noqa: F821
      .filter(f"batch_id = '{BACKFILL_BATCH}'")
      .select("batch_id", "layer", "table_name", "parameter", "load_type",
              "rows_read", "rows_inserted", "rows_updated", "status")
      .display())

batch_id,layer,table_name,parameter,load_type,rows_read,rows_inserted,rows_updated,status
backfill_PL_2023_demo,Raw-to-Bronze,bronze_fd_matches,/Volumes/workspace/default/football_lakehouse/raw/football-data/full_load/matches_PL_2023.json,FULL,380,0,380,SUCCESS


## 6.3 An incremental load

The standings and scorers feeds are **daily snapshots**: the same
competitions re-captured each day, keyed by `snapshot_date`, so history
accumulates instead of being overwritten. That is the incremental path,
and it is driven entirely by `--snapshot-date` + `--load-type
INCREMENTAL`.

`scripts/daily_snapshot.py` fetches a new day from the API and then calls
exactly the two commands below. The API key is not in this notebook, so
this re-plays the existing payload under a later snapshot date - the load
path, the parameters, the merge keys and the audit rows are the real
ones; only the arrival of the file is simulated.

Silver reads **only the new date's partition**, not the whole table.

In [0]:
INC_DATE = "2026-09-28"          # the "next day" snapshot
inc_batch = new_batch_id("incremental_demo")

std_bronze = config.table_path("bronze", "bronze_fd_standings")
std_silver = config.table_path("silver", "silver_standings")
b_before = spark.read.format("delta").load(std_bronze).count()   # noqa: F821
s_before = spark.read.format("delta").load(std_silver).count()   # noqa: F821
print(f"before:  bronze_fd_standings={b_before:,}   silver_standings={s_before:,}")

print(f"\n$ ... --feed fd_standings --load-type INCREMENTAL --snapshot-date {INC_DATE}")
raw_to_bronze.run_feed(spark, "fd_standings",  # noqa: F821
                       f"{RAW_URI}/football-data/full_load/standings_*.json",
                       "INCREMENTAL", inc_batch, INC_DATE)

print(f"\n$ ... --table silver_standings --load-type INCREMENTAL --snapshot-date {INC_DATE}")
bronze_to_silver.run_table(spark, "silver_standings", inc_batch, "INCREMENTAL", INC_DATE)  # noqa: F821

b_after = spark.read.format("delta").load(std_bronze).count()    # noqa: F821
s_after = spark.read.format("delta").load(std_silver).count()    # noqa: F821
print(f"\nafter :  bronze_fd_standings={b_after:,}   silver_standings={s_after:,}")
print(f"new rows: bronze +{b_after-b_before:,}   silver +{s_after-s_before:,}")

# On the first run this is a pure append of a new date. On a re-run the date is
# already there and MERGE updates it in place, so the test is "the new snapshot
# is present and the earlier days are still intact", not "the count grew".
dates = [str(r[0]) for r in spark.read.format("delta").load(std_silver)  # noqa: F821
         .select("snapshot_date").distinct().orderBy("snapshot_date").collect()]
print("snapshot dates held:", ", ".join(dates))
print("INCREMENTAL SNAPSHOT PRESENT:", "PASS" if INC_DATE in dates else "FAIL")
print("EARLIER SNAPSHOT RETAINED  :", "PASS" if SNAPSHOT_DATE in dates else "FAIL")

before:  bronze_fd_standings=212   silver_standings=212

$ ... --feed fd_standings --load-type INCREMENTAL --snapshot-date 2026-09-28
[bronze_fd_standings] read=212 inserted=212 updated=0 quarantined=0

$ ... --table silver_standings --load-type INCREMENTAL --snapshot-date 2026-09-28
[silver_standings] read=212 inserted=212 updated=0 quarantined=0

after :  bronze_fd_standings=424   silver_standings=424
new rows: bronze +212   silver +212
snapshot dates held: 2026-09-27, 2026-09-28
INCREMENTAL SNAPSHOT PRESENT: PASS
EARLIER SNAPSHOT RETAINED  : PASS


In [0]:
# the table now holds more than one day, which is the point of the snapshot design
(spark.read.format("delta").load(std_silver)  # noqa: F821
      .groupBy("snapshot_date").count().orderBy("snapshot_date").display())

snapshot_date,count
2026-09-27,212
2026-09-28,212


## 6.4 Schema drift, live

The real source data is clean, so the quarantine path never fires during
a normal run. This writes a deliberately broken file and loads it, to
show the four failure modes being handled rather than crashing the batch:

| case | corruption | how it should be handled |
|---|---|---|
| 1 | primary key is null | **quarantined** at Bronze - a row with no key cannot be merged |
| 2 | `utcDate` is no longer a timestamp | **quarantined** at Silver - the cast fails, the row is parked |
| 3 | upstream added a field we never declared | **dropped** - schema-on-read ignores it, the row loads |
| 4 | an int field arrives as a string | **nulled** - the declared type wins, the row loads |

The common requirement across all four: `status=SUCCESS`, the two clean
records load, nothing silently disappears, and the table's declared types
never change because of what arrived. Cases 3 and 4 deliberately do *not*
quarantine - the record is still usable, so rejecting it would lose good
data.

In [0]:
import json

DRIFT_DIR = f"{RAW}/drift_demo"
DRIFT_URI = f"{RAW_URI}/drift_demo"
os.makedirs(DRIFT_DIR, exist_ok=True)


def _good(mid, utc="2026-09-20T14:00:00Z"):
    return {
        "id": mid, "utcDate": utc, "status": "FINISHED", "matchday": 5,
        "stage": "REGULAR_SEASON", "group": None, "lastUpdated": "2026-09-27T00:20:33Z",
        "competition": {"id": 2021, "name": "Premier League", "code": "PL", "type": "LEAGUE"},
        "season": {"id": 2502, "startDate": "2026-08-21", "endDate": "2027-05-30",
                   "currentMatchday": 6},
        "homeTeam": {"id": 1, "name": "Drift Home FC", "tla": "DHM"},
        "awayTeam": {"id": 2, "name": "Drift Away FC", "tla": "DAW"},
        "score": {"winner": "HOME_TEAM", "duration": "REGULAR",
                  "fullTime": {"home": 2, "away": 1}, "halfTime": {"home": 1, "away": 0}},
    }


rows = [_good(999_000_001), _good(999_000_002)]
rows.append(_good(None))                                        # 1 null PK
rows.append(_good(999_000_003, utc="not-a-timestamp-at-all"))    # 2 type drift
_m = _good(999_000_004); _m["brandNewFieldFromUpstream"] = {"nested": "v", "n": 42}
rows.append(_m)                                                  # 3 undeclared column
_m = _good(999_000_005); _m["score"]["fullTime"]["home"] = "two"
rows.append(_m)                                                  # 4 int -> string

with open(f"{DRIFT_DIR}/matches_DRIFT_9999.json", "w") as f:
    json.dump({"competition": {"id": 2021, "name": "Premier League",
                               "code": "PL", "type": "LEAGUE"},
               "resultSet": {"count": len(rows), "played": len(rows)},
               "matches": rows}, f)
print(f"wrote {len(rows)} records, 4 of them deliberately broken, to {DRIFT_DIR}")

wrote 6 records, 4 of them deliberately broken, to /Volumes/workspace/default/football_lakehouse/raw/drift_demo


In [0]:
drift_batch = new_batch_id("drift_demo")
print("loading the corrupt file through the normal Bronze loader\n")
raw_to_bronze.run_feed(spark, "fd_matches", f"{DRIFT_URI}/matches_DRIFT_9999.json",  # noqa: F821
                       "FULL", drift_batch, None)
print("\nthe batch completed - a broken record did not take the run down\n")

print("then Silver over the same data (this is where the casts happen):")
bronze_to_silver.run_table(spark, "silver_matches", drift_batch, "FULL", None)  # noqa: F821

loading the corrupt file through the normal Bronze loader

[bronze_fd_matches] read=6 inserted=5 updated=0 quarantined=1

the batch completed - a broken record did not take the run down

then Silver over the same data (this is where the casts happen):
[silver_matches] read=14005 inserted=4 updated=14001 quarantined=1


In [0]:
for tbl in ("bronze_fd_matches", "silver_matches"):
    try:
        q = (spark.read.format("delta").load(config.table_path("quarantine", tbl))  # noqa: F821
                  .filter(f"batch_id = '{drift_batch}'"))
        n = q.count()
        print(f"\n{tbl}: {n} record(s) quarantined")
        if n:
            q.groupBy("quarantine_reason").count().show(truncate=False)
            q.select([c for c in q.columns if c in
                      ("match_id", "utc_date", "quarantine_reason", "quarantined_at")]).show(truncate=False)
    except Exception as e:  # noqa: BLE001
        print(f"\n{tbl}: no quarantine table ({e})")


bronze_fd_matches: 1 record(s) quarantined
+-------------------+-----+
|quarantine_reason  |count|
+-------------------+-----+
|null_required_field|1    |
+-------------------+-----+

+--------+--------------------+-------------------+--------------------------+
|match_id|utc_date            |quarantine_reason  |quarantined_at            |
+--------+--------------------+-------------------+--------------------------+
|NULL    |2026-09-20T14:00:00Z|null_required_field|2026-10-10 13:19:06.903703|
+--------+--------------------+-------------------+--------------------------+


silver_matches: 1 record(s) quarantined
+------------------------+-----+
|quarantine_reason       |count|
+------------------------+-----+
|cast_failed_schema_drift|1    |
+------------------------+-----+

+---------+----------------------+------------------------+--------------------------+
|match_id |utc_date              |quarantine_reason       |quarantined_at            |
+---------+----------------------+----

In [0]:
# per-record outcome, so the four cases can be read off directly
demo_ids = [999000001, 999000002, 999000003, 999000004, 999000005]
loaded = {r[0] for r in spark.read.format("delta").load(bronze_matches)  # noqa: F821
          .filter(F.col("match_id").isin(demo_ids)).select("match_id").collect()}
sm = spark.read.format("delta").load(config.table_path("silver", "silver_matches"))  # noqa: F821
in_silver = {r[0] for r in sm.filter(F.col("match_id").isin(demo_ids)).select("match_id").collect()}

print(f"{'record':>12}  {'case':34} {'bronze':>8} {'silver':>8}")
cases = {999000001: "clean control", 999000002: "clean control",
         999000003: "2 - utcDate not a timestamp", 999000004: "3 - undeclared column",
         999000005: "4 - int arrived as string"}
for mid in demo_ids:
    print(f"{mid:>12}  {cases[mid]:34} {'yes' if mid in loaded else 'no':>8} "
          f"{'yes' if mid in in_silver else 'QUARANTINED':>8}")
print(f"\n{'(null)':>12}  {'1 - null primary key':34} {'QUARANTINED at Bronze':>8}")
print("\nscore for record 999000005 (the string 'two' against a declared int):")
spark.read.format("delta").load(bronze_matches).filter("match_id = 999000005") \
     .select("match_id", "full_time_home", "full_time_away",
             "half_time_home", "half_time_away").show()  # noqa: F821

      record  case                                 bronze   silver
   999000001  clean control                           yes      yes
   999000002  clean control                           yes      yes
   999000003  2 - utcDate not a timestamp             yes QUARANTINED
   999000004  3 - undeclared column                   yes      yes
   999000005  4 - int arrived as string               yes      yes

      (null)  1 - null primary key               QUARANTINED at Bronze

score for record 999000005 (the string 'two' against a declared int):
+---------+--------------+--------------+--------------+--------------+
| match_id|full_time_home|full_time_away|half_time_home|half_time_away|
+---------+--------------+--------------+--------------+--------------+
|999000005|          NULL|             1|             1|             0|
+---------+--------------+--------------+--------------+--------------+



### The undeclared column

Case 3 is the interesting one: the record is *valid*, it just carries a
field we never declared. Because the read is schema-on-read, the unknown
field is dropped rather than changing the table, and the row loads
normally. The pipeline keeps working through an upstream addition without
a migration and without a surprise column appearing in Silver.

In [0]:
print("record 999000004 carried brandNewFieldFromUpstream. Did it load, and did the column leak in?\n")
bm = spark.read.format("delta").load(bronze_matches)  # noqa: F821
print("loaded :", bm.filter("match_id = 999000004").count() == 1)
print("leaked :", any("brandnew" in c.lower() for c in bm.columns))
print("\ncolumns:", ", ".join(bm.columns))

record 999000004 carried brandNewFieldFromUpstream. Did it load, and did the column leak in?

loaded : True
leaked : False

columns: match_id, competition_code, competition_id, competition_name, season_id, season_start_date, season_end_date, season_current_matchday, utc_date, status, matchday, stage, group_name, source_last_updated, home_team_id, home_team_name, home_team_tla, away_team_id, away_team_name, away_team_tla, score_winner, score_duration, full_time_home, full_time_away, half_time_home, half_time_away, referees, source_file, load_timestamp, batch_id, load_type


### Clean up the demo rows

The drift records were synthetic, so remove them. This also exercises
Delta `DELETE`, which a plain parquet directory could not do.

In [0]:
from delta.tables import DeltaTable

for layer, tbl in (("bronze", "bronze_fd_matches"), ("silver", "silver_matches")):
    p = config.table_path(layer, tbl)
    DeltaTable.forPath(spark, p).delete("match_id >= 999000000")  # noqa: F821
    df = spark.read.format("delta").load(p)  # noqa: F821
    left = df.filter("match_id >= 999000000").count()
    print(f"{tbl}: {left} demo rows remaining (expected 0), {df.count():,} real rows intact")

# and remove the corrupt source file so a later full run does not re-ingest it
import shutil
shutil.rmtree(DRIFT_DIR, ignore_errors=True)
print("removed", DRIFT_DIR)

bronze_fd_matches: 0 demo rows remaining (expected 0), 14,001 real rows intact
silver_matches: 0 demo rows remaining (expected 0), 14,001 real rows intact
removed /Volumes/workspace/default/football_lakehouse/raw/drift_demo


## 6.5 Final audit log

One row per table per run, with the parameter it was given, the load
type, timings, row counts and status. Both FULL and INCREMENTAL now
appear, and the quarantined counts are non-zero for the drift batch.

In [0]:
log = spark.read.format("delta").load(config.EXECUTION_LOG_PATH)  # noqa: F821
print("total audit rows:", log.count())
log.groupBy("load_type", "status").count().orderBy("load_type").display()

total audit rows: 64


load_type,status,count
FULL,SUCCESS,59
FULL,FAILED,3
INCREMENTAL,SUCCESS,2


In [0]:
(log.select("start_time", "layer", "table_name", "parameter", "load_type", "batch_id",
            "status", "rows_read", "rows_inserted", "rows_updated", "rows_quarantined",
            "duration_seconds")
    .orderBy(F.col("start_time").desc())  # noqa: F821
    .display())

start_time,layer,table_name,parameter,load_type,batch_id,status,rows_read,rows_inserted,rows_updated,rows_quarantined,duration_seconds
2026-10-10T13:19:17.215Z,Bronze-to-Silver,silver_matches,ALL,FULL,drift_demo_20261010T131905_2bd277,SUCCESS,14005,4,14001,1,10
2026-10-10T13:19:05.600Z,Raw-to-Bronze,bronze_fd_matches,/Volumes/workspace/default/football_lakehouse/raw/drift_demo/matches_DRIFT_9999.json,FULL,drift_demo_20261010T131905_2bd277,SUCCESS,6,5,0,1,10
2026-10-10T13:18:52.851Z,Bronze-to-Silver,silver_standings,2026-09-28,INCREMENTAL,incremental_demo_20261010T131842_bc599c,SUCCESS,212,212,0,0,6
2026-10-10T13:18:43.876Z,Raw-to-Bronze,bronze_fd_standings,/Volumes/workspace/default/football_lakehouse/raw/football-data/full_load/standings_*.json,INCREMENTAL,incremental_demo_20261010T131842_bc599c,SUCCESS,212,212,0,0,7
2026-10-10T13:18:30.434Z,Raw-to-Bronze,bronze_fd_matches,/Volumes/workspace/default/football_lakehouse/raw/football-data/full_load/matches_PL_2023.json,FULL,backfill_PL_2023_demo,SUCCESS,380,0,380,0,9
2026-10-10T13:17:58.772Z,Bronze-to-Silver,silver_events,ALL,FULL,silver_p2_20261010T131705_19f152,SUCCESS,267255,0,267255,0,10
2026-10-10T13:17:50.688Z,Bronze-to-Silver,silver_lineups,ALL,FULL,silver_p2_20261010T131705_19f152,SUCCESS,2620,0,2620,0,7
2026-10-10T13:17:41.331Z,Bronze-to-Silver,silver_sb_matches,ALL,FULL,silver_p2_20261010T131705_19f152,SUCCESS,66,0,66,0,8
2026-10-10T13:17:32.544Z,Bronze-to-Silver,silver_teams,ALL,FULL,silver_p2_20261010T131705_19f152,SUCCESS,284,0,284,0,7
2026-10-10T13:17:23.702Z,Bronze-to-Silver,silver_scorers,2026-09-27,FULL,silver_p2_20261010T131705_19f152,SUCCESS,600,0,600,0,7


### Checklist

Everything above ran in this workspace, against Delta tables in a Unity
Catalog volume. Export this notebook with its outputs
(**File -> Export -> HTML**) and commit the file, or screen-record
sections 2-6.